# CardioIA — diagnóstico visual acadêmico com MLP

Notebook do **Ir Além 2**. O objetivo é classificar imagens de ECG em **normal** e **anormal** usando uma rede Perceptron Multicamadas implementada com Keras.

> O experimento é educacional, não possui validação clínica e não produz diagnóstico.

## 1. Governança antes do treinamento

A fonte curada da Fase 1 possui 120 imagens, com 30 exames por classe original. Para o experimento binário, são usados os 30 exames normais e uma seleção determinística de 10 exames de cada uma das três classes anormais, totalizando 60 imagens equilibradas. Esse equilíbrio é experimental e não representa prevalência. Os hashes garantem que não há cópias binárias na amostra.

A fonte não fornece identificador de paciente. Portanto, conseguimos impedir que o mesmo exame duplicado apareça em treino e teste, mas **não podemos garantir separação por indivíduo**. O balanceamento da amostra também não representa prevalência real.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

RAIZ = Path.cwd()
if not (RAIZ / "fase2").exists():
    RAIZ = Path.cwd().parents[1]
sys.path.insert(0, str(RAIZ / "fase2" / "src"))

from treinar_mlp_ecg import (
    carregar_pixels,
    criar_mlp,
    dividir_dados,
    inventariar_imagens,
    treinar_mlp,
)

## 2. Inventário e classificação binária

A classe `normal` permanece normal. Infarto do miocárdio, histórico de infarto e batimento anormal são agrupados como `anormal`, sem combinar estes exames com os registros tabulares ou textuais.

In [2]:
inventario = inventariar_imagens()
display(pd.crosstab(inventario["classe_original"], inventario["classe_binaria"]))
display(pd.DataFrame({
    "total": [len(inventario)],
    "hashes_unicos": [inventario["hash_sha256"].nunique()],
    "duplicadas": [int(inventario["hash_sha256"].duplicated().sum())],
}))

classe_binaria,anormal,normal
classe_original,,
abnormal_heartbeat,10,0
history_mi,10,0
myocardial_infarction,10,0
normal,0,30


,total,hashes_unicos,duplicadas
0,60,60,0


## 3. Pré-processamento

Cada imagem é convertida para tons de cinza, redimensionada para 64 × 64 pixels, normalizada para o intervalo 0–1 e achatada em um vetor com 4.096 entradas, formato adequado para uma MLP.

In [3]:
pixels = carregar_pixels(inventario)
display(pd.DataFrame({
    "amostras": [pixels.shape[0]],
    "atributos_por_imagem": [pixels.shape[1]],
    "valor_minimo": [pixels.min()],
    "valor_maximo": [pixels.max()],
}))

,amostras,atributos_por_imagem,valor_minimo,valor_maximo
0,60,4096,0.0,0.537255


## 4. Divisão estratificada e verificação de vazamento

A divisão utiliza 80% dos exames para treino e 20% para teste. A estratificação preserva a proporção das classes binárias.

In [4]:
x_treino, x_teste, y_treino, y_teste, treino_idx, teste_idx = dividir_dados(
    inventario,
    pixels,
)
assert not set(inventario.iloc[treino_idx]["hash_sha256"]).intersection(
    inventario.iloc[teste_idx]["hash_sha256"]
)
display(pd.DataFrame({
    "conjunto": ["treino", "teste"],
    "imagens": [len(y_treino), len(y_teste)],
    "normais": [(y_treino == 0).sum(), (y_teste == 0).sum()],
    "anormais": [(y_treino == 1).sum(), (y_teste == 1).sum()],
}))

,conjunto,imagens,normais,anormais
0,treino,48,24,24
1,teste,12,6,6


## 5. Arquitetura MLP com Keras

A rede recebe o vetor de pixels, utiliza duas camadas densas com ativação ReLU, dropout para regularização e uma saída sigmoide para classificação binária.

In [5]:
modelo_demonstracao = criar_mlp(64 * 64)
modelo_demonstracao.summary()

Model: "mlp_ecg_binaria"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │       524,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)      

## 6. Treinamento e avaliação

São usados pesos de classe para reduzir o efeito do desbalanceamento binário. O early stopping restaura os melhores pesos observados na validação.

In [6]:
modelo, historico, inventario, metricas, previsoes, y_teste, y_predito = treinar_mlp()
resumo = {
    chave: valor
    for chave, valor in metricas.items()
    if chave != "relatorio"
}
display(pd.DataFrame([resumo]).style.format({
    "acuracia": "{:.3f}",
    "acuracia_balanceada": "{:.3f}",
    "precisao_anormal": "{:.3f}",
    "recall_anormal": "{:.3f}",
    "f1_anormal": "{:.3f}",
    "roc_auc": "{:.3f}",
}))
display(pd.DataFrame(metricas["relatorio"]).T)

,limiar_selecionado_na_validacao,n_ajuste_apos_aumento,acuracia,acuracia_balanceada,precisao_anormal,recall_anormal,f1_anormal,roc_auc,n_treino,n_teste,epocas_executadas
0,0.500000,180,0.417,0.417,0.333,0.167,0.222,0.667,48,12,17


,precision,recall,f1-score,support
normal,0.444444,0.666667,0.533333,6.000000
anormal,0.333333,0.166667,0.222222,6.000000
accuracy,0.416667,0.416667,0.416667,0.416667
macro avg,0.388889,0.416667,0.377778,12.000000
weighted avg,0.388889,0.416667,0.377778,12.000000


In [7]:
pd.DataFrame(historico.history)[["loss", "val_loss"]].plot(
    title="Perda durante o treinamento"
)
plt.xlabel("Época")
plt.ylabel("Binary cross-entropy")
plt.show()

display(previsoes.head(10))

,arquivo,classe_original,classe_binaria,hash_sha256,probabilidade_anormal,predicao
24,normal_026.jpg,normal,normal,f7df45104dbfef8adbeb9d4464c7877a000b6c6b1f6c0b...,0.249683,normal
58,mi_026.jpg,myocardial_infarction,anormal,324e8ca46fd023520f36c3427dde92aebc1419c527d6ae...,0.290485,normal
25,mi_030.jpg,myocardial_infarction,anormal,fff32301dcd37c586b564c07b6bce50b71abaf1d1af29f...,0.669935,anormal
16,normal_009.jpg,normal,normal,0a735169e983cbc5a9fb0c0eec4f6062b2349576f1f050...,0.592871,anormal
29,ahb_028.jpg,abnormal_heartbeat,anormal,c955a845b354aa0a15cadaa8a9239873081a863dffa5e0...,0.403123,normal
30,normal_027.jpg,normal,normal,254a0e3e64d9569a3697805c742c902ae6f892026fc458...,0.337882,normal
14,mi_011.jpg,myocardial_infarction,anormal,03a9720a18db30facbecd92c6ab8c86b16e88622294f02...,0.338538,normal
17,normal_007.jpg,normal,normal,c815b51e93aec829b8633e88b498e8a2e8a1195406b838...,0.589136,anormal
48,hmi_030.jpg,history_mi,anormal,3fc44abb037e4a54a2318c1e1f790c823dc5668d0fa472...,0.422513,normal
41,normal_002.jpg,normal,normal,5e6100631abcf13ff32b80eb0d09dc0d1c065bf60a5c3e...,0.317420,normal


## 7. Conclusão responsável

A atividade demonstra pré-processamento de imagens, implementação de MLP com Keras, treinamento e avaliação. Os resultados devem ser lidos apenas como desempenho nesta pequena amostra curada.

Limitações centrais:

- ausência de identificador de paciente;
- apenas 120 imagens;
- classes originais artificialmente balanceadas;
- origem populacional e tecnológica específica;
- MLP sobre pixels achatados, sem arquitetura especializada em visão;
- ausência de validação externa, prospectiva ou clínica.